# Finance Domain Model Comparison
**Goal:** Find if any finance-domain embedding model beats MiniLM on FiQA  
**Evaluation:** 30% of FiQA test queries (same 194 queries for all 3 models — fair comparison)  
**Models tested:**
- `all-MiniLM-L6-v2` — baseline (general purpose, 384-dim, 22M params)
- `FinLang/finance-embeddings-investopedia` — fine-tuned on Investopedia (768-dim, 109M params)
- `baconnier/Finance_embedding_small_en-V1.5` — finance triplet-trained (384-dim, 33M params)

**Metrics:** NDCG@10, MRR, Recall@10 (same as all previous notebooks)

## Cell 0 — Repo Root & Config

In [11]:
import sys, os
from pathlib import Path

_root = Path().resolve()
for _ in range(5):
    if (_root / 'config.py').exists():
        break
    _root = _root.parent

os.chdir(_root)
sys.path.insert(0, str(_root))

from config import FIQA_CORPUS, FIQA_QUERIES, FIQA_QRELS_TEST

COMPARISON_DIR = os.path.join('finrerank', 'ModelComparison_Results')
os.makedirs(COMPARISON_DIR, exist_ok=True)

print('Repo root    :', _root)
print('Results dir  :', COMPARISON_DIR)

Repo root    : /Users/momo/Desktop/GEN AI/Finsearch Project/FinSearch_Intent_Aware_Financial_Document_Intelligence-
Results dir  : finrerank/ModelComparison_Results


## Cell 1 — Install Dependencies

In [12]:
import sys
!{sys.executable} -m pip install -q sentence-transformers faiss-cpu pytrec-eval-terrier

## Cell 2 — Imports

In [13]:
import os, json
os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'

import numpy as np
import pandas as pd
import faiss
import pytrec_eval
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

print('Imports OK')

Imports OK


## Cell 3 — Load FiQA Corpus, Queries & Qrels

In [14]:
print('Loading corpus...')
corpus_df = pd.read_csv(FIQA_CORPUS, usecols=['_id', 'text'])
corpus_df = corpus_df.dropna(subset=['text']).reset_index(drop=True)
corpus_df['_id'] = corpus_df['_id'].astype(str)
print(f'Full corpus: {len(corpus_df):,} passages')

queries_df = pd.read_csv(FIQA_QUERIES, usecols=['_id', 'text'])
queries_df['_id'] = queries_df['_id'].astype(str)

qrels_df = pd.read_csv(FIQA_QRELS_TEST)
qrels_df.columns = [c.strip() for c in qrels_df.columns]
qrels_df = qrels_df.astype(str)
query_col  = [c for c in qrels_df.columns if 'query'  in c.lower()][0]
corpus_col = [c for c in qrels_df.columns if 'corpus' in c.lower()][0]
score_col  = [c for c in qrels_df.columns if 'score'  in c.lower()][0]
qrels_dict = {}
for _, row in qrels_df.iterrows():
    qid = str(row[query_col])
    did = str(row[corpus_col])
    qrels_dict.setdefault(qid, {})[did] = int(float(row[score_col]))

test_qids       = list(qrels_dict.keys())
test_queries_df = queries_df[queries_df['_id'].isin(test_qids)].reset_index(drop=True)
print(f'Test queries: {len(test_queries_df):,} | Qrels: {len(qrels_dict):,}')

# ── Stratified corpus reduction for fast model comparison ────────────────────
# Step 1: Keep ALL passages that appear in qrels (relevant docs must be present)
#         → guarantees fair evaluation — no relevant doc is dropped
# Step 2: Add random 8% of remaining passages as negatives
#         → model still has to distinguish relevant from irrelevant
# Result: ~6,000 passages instead of 57,600 → 10x faster encoding
#         Absolute metrics will be lower but RELATIVE ranking between models is valid

qrel_doc_ids = set(did for docs in qrels_dict.values() for did in docs.keys())
relevant_df  = corpus_df[corpus_df['_id'].isin(qrel_doc_ids)].reset_index(drop=True)
remaining_df = corpus_df[~corpus_df['_id'].isin(qrel_doc_ids)]
sampled_neg  = remaining_df.sample(frac=0.08, random_state=42).reset_index(drop=True)
corpus_df    = pd.concat([relevant_df, sampled_neg]).reset_index(drop=True)

corpus_ids        = corpus_df['_id'].tolist()
corpus_texts      = corpus_df['text'].tolist()
corpus_id_to_text = dict(zip(corpus_ids, corpus_texts))

print(f'\nReduced corpus  : {len(corpus_df):,} passages')
print(f'  Relevant docs : {len(relevant_df):,} (all qrel docs kept — fair evaluation)')
print(f'  Negatives     : {len(sampled_neg):,} (8% random sample)')
print(f'  Speedup       : ~{57600 // len(corpus_df)}x faster encoding')
print(f'  Note          : absolute metrics lower than full corpus but relative ranking valid')

# ── 30% query sample — same for all 3 models ─────────────────────────────────
sample_df    = test_queries_df.sample(frac=0.30, random_state=42).reset_index(drop=True)
sample_qids  = sample_df['_id'].tolist()
sample_texts = sample_df['text'].tolist()
sample_qrels = {qid: qrels_dict[qid] for qid in sample_qids if qid in qrels_dict}
print(f'\n30% query sample: {len(sample_df)} queries (same random_state=42 for all models)')

Loading corpus...
Full corpus: 57,600 passages
Test queries: 648 | Qrels: 648

Reduced corpus  : 6,177 passages
  Relevant docs : 1,705 (all qrel docs kept — fair evaluation)
  Negatives     : 4,472 (8% random sample)
  Speedup       : ~9x faster encoding
  Note          : absolute metrics lower than full corpus but relative ranking valid

30% query sample: 194 queries (same random_state=42 for all models)


## Cell 4 — Helper Functions
Shared by all 3 models — encode corpus, build FAISS, retrieve, evaluate.

In [15]:
def build_index(model, corpus_texts, batch_size=256, query_prefix=''):
    """Encode corpus and build FAISS IndexFlatIP (cosine via L2-norm)."""
    print(f'  Encoding {len(corpus_texts):,} passages (batch={batch_size})...')
    embeddings = model.encode(
        corpus_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)
    dim   = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    print(f'  Index built: dim={dim}  vectors={index.ntotal:,}')
    return index


def retrieve(model, index, query_texts, query_ids, top_k=10,
             batch_size=128, query_prefix=''):
    """Encode queries and search FAISS index."""
    prefixed = [query_prefix + q for q in query_texts] if query_prefix else query_texts
    q_embs   = model.encode(
        prefixed,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)
    scores_mat, idx_mat = index.search(q_embs, top_k)
    results = {}
    for i, qid in enumerate(query_ids):
        results[qid] = {
            corpus_ids[idx]: float(scores_mat[i, rank])
            for rank, idx in enumerate(idx_mat[i])
            if idx != -1
        }
    return results


def evaluate(results_dict, qrels_dict):
    """Compute NDCG@10, MRR, Recall@10 using pytrec_eval."""
    evaluator = pytrec_eval.RelevanceEvaluator(
        qrels_dict, {'ndcg_cut_10', 'recip_rank', 'recall_10'}
    )
    res = evaluator.evaluate(results_dict)
    return {
        'NDCG@10':   round(np.mean([v['ndcg_cut_10'] for v in res.values()]), 4),
        'MRR':       round(np.mean([v['recip_rank']  for v in res.values()]), 4),
        'Recall@10': round(np.mean([v['recall_10']   for v in res.values()]), 4),
        'num_queries': len(res),
    }

print('Helper functions defined.')

Helper functions defined.


## Cell 5 — Model 1: MiniLM (Baseline Reference)
`all-MiniLM-L6-v2` — general purpose, 384-dim, 22M params  
Evaluated on same 30% sample for fair comparison.

In [16]:
print('=' * 55)
print('Model 1: all-MiniLM-L6-v2 (Baseline)')
print('=' * 55)

minilm = SentenceTransformer('all-MiniLM-L6-v2')
print(f'  Dim: {minilm.get_sentence_embedding_dimension()}  Params: ~22M')

print('Building index...')
minilm_index = build_index(minilm, corpus_texts, batch_size=256)

print('Retrieving on 30% sample...')
minilm_results = retrieve(minilm, minilm_index, sample_texts, sample_qids, top_k=10)

minilm_metrics = evaluate(minilm_results, sample_qrels)
print(f'\nMiniLM Results ({minilm_metrics["num_queries"]} queries):')
print(f'  NDCG@10   : {minilm_metrics["NDCG@10"]}')
print(f'  MRR       : {minilm_metrics["MRR"]}')
print(f'  Recall@10 : {minilm_metrics["Recall@10"]}')

Model 1: all-MiniLM-L6-v2 (Baseline)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9573.49it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Dim: 384  Params: ~22M
Building index...
  Encoding 6,177 passages (batch=256)...


Batches: 100%|██████████| 25/25 [00:40<00:00,  1.64s/it]


  Index built: dim=384  vectors=6,177
Retrieving on 30% sample...


Batches: 100%|██████████| 2/2 [00:00<00:00, 15.04it/s]


MiniLM Results (194 queries):
  NDCG@10   : 0.5821
  MRR       : 0.6721
  Recall@10 : 0.6468


## Cell 6 — Model 2: BGE-Base-v1.5\n`BAAI/bge-base-en-v1.5` — 768-dim, 109M params  \nBGE Base (not Large) — strong retrieval model, trained on diverse text including finance.  \nUses query prefix: `'Represent this sentence for searching relevant passages: '`

In [17]:
print('=' * 55)
print('Model 2: BAAI/bge-base-en-v1.5')
print('=' * 55)

BGE_PREFIX = 'Represent this sentence for searching relevant passages: '

bge_base = SentenceTransformer('BAAI/bge-base-en-v1.5')
print(f'  Dim: {bge_base.get_sentence_embedding_dimension()}  Params: ~109M')

print('Building index...')
bge_base_index = build_index(bge_base, corpus_texts, batch_size=128)

print('Retrieving on 30% sample...')
bge_base_results = retrieve(bge_base, bge_base_index, sample_texts, sample_qids,
                             top_k=10, query_prefix=BGE_PREFIX)

bge_base_metrics = evaluate(bge_base_results, sample_qrels)
print(f'\nBGE-Base Results ({bge_base_metrics["num_queries"]} queries):')
print(f'  NDCG@10   : {bge_base_metrics["NDCG@10"]}')
print(f'  MRR       : {bge_base_metrics["MRR"]}')
print(f'  Recall@10 : {bge_base_metrics["Recall@10"]}')

Model 2: BAAI/bge-base-en-v1.5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8307.21it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Dim: 768  Params: ~109M
Building index...
  Encoding 6,177 passages (batch=128)...


Batches: 100%|██████████| 49/49 [31:18<00:00, 38.34s/it]   


  Index built: dim=768  vectors=6,177
Retrieving on 30% sample...


Batches: 100%|██████████| 2/2 [00:00<00:00,  2.32it/s]


BGE-Base Results (194 queries):
  NDCG@10   : 0.5817
  MRR       : 0.6549
  Recall@10 : 0.657


In [18]:
del bge_base_index
import gc; gc.collect()

6453

## Cell 7 — Model 3: E5-Small-v2\n`intfloat/e5-small-v2` — 384-dim, 33M params  \nTrained specifically for retrieval tasks with query/passage prefixes.  \nSame size as MiniLM but retrieval-optimized — uses `'query: '` prefix for queries, `'passage: '` for corpus.

In [19]:
print('=' * 55)
print('Model 3: intfloat/e5-small-v2')
print('=' * 55)

E5_QUERY_PREFIX   = 'query: '
E5_PASSAGE_PREFIX = 'passage: '

e5small = SentenceTransformer('intfloat/e5-small-v2')
print(f'  Dim: {e5small.get_sentence_embedding_dimension()}  Params: ~33M')

# E5 requires 'passage: ' prefix on corpus texts
print('Building index (with passage: prefix)...')
corpus_texts_e5 = [E5_PASSAGE_PREFIX + t for t in corpus_texts]
e5small_index   = build_index(e5small, corpus_texts_e5, batch_size=256)

# E5 requires 'query: ' prefix on query texts
print('Retrieving on 30% sample (with query: prefix)...')
e5small_results = retrieve(e5small, e5small_index, sample_texts, sample_qids,
                            top_k=10, query_prefix=E5_QUERY_PREFIX)

e5small_metrics = evaluate(e5small_results, sample_qrels)
print(f'\nE5-Small Results ({e5small_metrics["num_queries"]} queries):')
print(f'  NDCG@10   : {e5small_metrics["NDCG@10"]}')
print(f'  MRR       : {e5small_metrics["MRR"]}')
print(f'  Recall@10 : {e5small_metrics["Recall@10"]}')

Model 3: intfloat/e5-small-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8639.72it/s]
BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Dim: 384  Params: ~33M
Building index (with passage: prefix)...
  Encoding 6,177 passages (batch=256)...


Batches: 100%|██████████| 25/25 [51:38<00:00, 123.93s/it] 


  Index built: dim=384  vectors=6,177
Retrieving on 30% sample (with query: prefix)...


Batches: 100%|██████████| 2/2 [00:00<00:00,  8.63it/s]


E5-Small Results (194 queries):
  NDCG@10   : 0.5634
  MRR       : 0.6394
  Recall@10 : 0.6323


In [20]:
del e5small_index
import gc; gc.collect()


4039

## Cell 8 — Side-by-Side Comparison & Save Results

## Cell 7b — Model 4: BGE-Large-v1.5
`BAAI/bge-large-en-v1.5` — 1024-dim, 335M params  
Same model used in FinSearch_Optimized.ipynb — was too slow on full corpus (6 hrs).  
Now feasible with reduced corpus (~6,000 passages). Fair comparison on same 30% queries.

In [21]:
print('=' * 55)
print('Model 4: BAAI/bge-large-en-v1.5')
print('=' * 55)
print('Previously took 6 hours on full corpus — now fast on reduced corpus')

BGE_LARGE_PREFIX = 'Represent this sentence for searching relevant passages: '

bge_large = SentenceTransformer('BAAI/bge-large-en-v1.5')
print(f'  Dim: {bge_large.get_sentence_embedding_dimension()}  Params: ~335M')

print('Building index on reduced corpus...')
bge_large_index = build_index(bge_large, corpus_texts, batch_size=64)

print('Retrieving on 30% sample...')
bge_large_results = retrieve(bge_large, bge_large_index, sample_texts, sample_qids,
                              top_k=10, query_prefix=BGE_LARGE_PREFIX)

bge_large_metrics = evaluate(bge_large_results, sample_qrels)
print(f'\nBGE-Large Results ({bge_large_metrics["num_queries"]} queries):')
print(f'  NDCG@10   : {bge_large_metrics["NDCG@10"]}')
print(f'  MRR       : {bge_large_metrics["MRR"]}')
print(f'  Recall@10 : {bge_large_metrics["Recall@10"]}')

Model 4: BAAI/bge-large-en-v1.5
Previously took 6 hours on full corpus — now fast on reduced corpus


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 7747.34it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Dim: 1024  Params: ~335M
Building index on reduced corpus...
  Encoding 6,177 passages (batch=64)...


Batches: 100%|██████████| 97/97 [30:33<00:00, 18.90s/it]  


  Index built: dim=1024  vectors=6,177
Retrieving on 30% sample...


Batches: 100%|██████████| 2/2 [15:19<00:00, 459.96s/it]



BGE-Large Results (194 queries):
  NDCG@10   : 0.6355
  MRR       : 0.699
  Recall@10 : 0.7258


In [22]:
MINILM_FULL = {'NDCG@10': 0.3687, 'MRR': 0.4451, 'Recall@10': 0.4413}

models = [
    ('all-MiniLM-L6-v2 (full 648)',   '22M',  384,  MINILM_FULL['NDCG@10'],       MINILM_FULL['MRR'],       MINILM_FULL['Recall@10'],       648),
    ('all-MiniLM-L6-v2 (30% sample)', '22M',  384,  minilm_metrics['NDCG@10'],    minilm_metrics['MRR'],    minilm_metrics['Recall@10'],    minilm_metrics['num_queries']),
    ('BAAI/bge-base-en-v1.5',         '109M', 768,  bge_base_metrics['NDCG@10'],  bge_base_metrics['MRR'],  bge_base_metrics['Recall@10'],  bge_base_metrics['num_queries']),
    ('intfloat/e5-small-v2',          '33M',  384,  e5small_metrics['NDCG@10'],   e5small_metrics['MRR'],   e5small_metrics['Recall@10'],   e5small_metrics['num_queries']),
    ('BAAI/bge-large-en-v1.5',        '335M', 1024, bge_large_metrics['NDCG@10'], bge_large_metrics['MRR'], bge_large_metrics['Recall@10'], bge_large_metrics['num_queries']),
]

best_ndcg = max(m[3] for m in models)

print()
print('=' * 85)
print('   EMBEDDING MODEL COMPARISON — FiQA (reduced corpus + 30% same queries)')
print('=' * 85)
header = '  {:<38} {:>6} {:>5} {:>8} {:>8} {:>10}'.format('Model','Params','Dim','NDCG@10','MRR','Recall@10')
print(header)
print('-' * 85)
for name, params, dim, ndcg, mrr, recall, n in models:
    tag = '  <- BEST' if ndcg == best_ndcg else ''
    ref = '  (reference)' if '648' in name else ''
    row = '  {:<38} {:>6} {:>5} {:>8.4f} {:>8.4f} {:>10.4f}{}{}'.format(
        name, params, dim, ndcg, mrr, recall, tag, ref)
    print(row)
print('=' * 85)
print()
print('Note: absolute metrics inflated vs full corpus — only relative ranking matters')
print()

sample_results = [
    ('MiniLM',     minilm_metrics['NDCG@10']),
    ('BGE-Base',   bge_base_metrics['NDCG@10']),
    ('E5-Small',   e5small_metrics['NDCG@10']),
    ('BGE-Large',  bge_large_metrics['NDCG@10']),
]
winner = max(sample_results, key=lambda x: x[1])
print('Winner on reduced corpus : {} (NDCG@10 = {})'.format(winner[0], winner[1]))
print()
if winner[0] == 'MiniLM':
    print('Conclusion: MiniLM still wins — general purpose model fits FiQA informal Q&A better.')
    print('  Fine-tune MiniLM on FiQA qrels for further improvement.')
elif winner[0] == 'BGE-Large':
    print('Conclusion: BGE-Large wins on reduced corpus.')
    print('  Previously failed on full corpus due to CPU time — not model quality.')
    print('  Consider BGE-Large on GPU for production pipeline.')
elif winner[0] == 'BGE-Base':
    print('Conclusion: BGE-Base beats MiniLM — good balance of quality and speed.')
else:
    print('Conclusion: E5-Small wins — retrieval-optimized training helps on FiQA.')

# Save updated results
results_out = {
    'evaluation': 'FiQA reduced corpus + 30% sample (random_state=42)',
    'corpus': 'Stratified: all qrel docs + 8% random negatives',
    'note': 'Absolute metrics inflated — relative ranking valid for model selection',
    'models': {
        'MiniLM-30%':  {k: minilm_metrics[k]    for k in ['NDCG@10','MRR','Recall@10','num_queries']},
        'BGE-Base':    {k: bge_base_metrics[k]   for k in ['NDCG@10','MRR','Recall@10','num_queries']},
        'E5-Small':    {k: e5small_metrics[k]    for k in ['NDCG@10','MRR','Recall@10','num_queries']},
        'BGE-Large':   {k: bge_large_metrics[k]  for k in ['NDCG@10','MRR','Recall@10','num_queries']},
    },
    'MiniLM_full_648_reference': MINILM_FULL,
    'winner': winner[0],
}
with open(os.path.join(COMPARISON_DIR, 'model_comparison.json'), 'w') as f:
    json.dump(results_out, f, indent=2)
print('\nSaved: finrerank/ModelComparison_Results/model_comparison.json')


   EMBEDDING MODEL COMPARISON — FiQA (reduced corpus + 30% same queries)
  Model                                  Params   Dim  NDCG@10      MRR  Recall@10
-------------------------------------------------------------------------------------
  all-MiniLM-L6-v2 (full 648)               22M   384   0.3687   0.4451     0.4413  (reference)
  all-MiniLM-L6-v2 (30% sample)             22M   384   0.5821   0.6721     0.6468
  BAAI/bge-base-en-v1.5                    109M   768   0.5817   0.6549     0.6570
  intfloat/e5-small-v2                      33M   384   0.5634   0.6394     0.6323
  BAAI/bge-large-en-v1.5                   335M  1024   0.6355   0.6990     0.7258  <- BEST

Note: absolute metrics inflated vs full corpus — only relative ranking matters

Winner on reduced corpus : BGE-Large (NDCG@10 = 0.6355)

Conclusion: BGE-Large wins on reduced corpus.
  Previously failed on full corpus due to CPU time — not model quality.
  Consider BGE-Large on GPU for production pipeline.

Saved: fin